# Granite ASR — GPU Inference on Colab

This notebook runs the `granite-asr` package on a Colab GPU runtime.

**Before running:** Select `Runtime → Change runtime type → T4 GPU` (or better).

## 1. Install dependencies

In [ ]:
# Install CUDA PyTorch (already present on Colab GPU runtimes, but pinning for safety)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    print(result.stdout[-2000:] if result.stdout else "(no output)")

# PyTorch with CUDA — skip if already installed with CUDA support
import torch
if not torch.cuda.is_available():
    print("CUDA not available — installing CUDA-enabled PyTorch...")
    run("pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121 -q")
else:
    print(f"PyTorch {torch.__version__} already has CUDA support: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repo and install granite-asr with server extras
import os

if not os.path.exists("EEVA-edge"):
    run("git clone --depth 1 https://github.com/your-org/EEVA-edge.git")

run("pip install -e EEVA-edge/granite_asr[server] -q")
print("granite-asr installed.")

## 2. Download sample audio

We use a short LibriVox public-domain clip (~30 s). You can replace this with your own file in the next cell.

In [ ]:
SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/1/1f/Liber_Primus_cap_1.ogg"
SAMPLE_FILE = "sample.ogg"

if not os.path.exists(SAMPLE_FILE):
    run(f"wget -q '{SAMPLE_URL}' -O {SAMPLE_FILE}")

print(f"Sample audio saved to {SAMPLE_FILE}")

## 3. (Optional) Upload your own audio file

In [ ]:
from google.colab import files

USE_UPLOAD = False  # Set to True to upload your own file

if USE_UPLOAD:
    uploaded = files.upload()
    AUDIO_FILE = list(uploaded.keys())[0]
    print(f"Using uploaded file: {AUDIO_FILE}")
else:
    AUDIO_FILE = SAMPLE_FILE
    print(f"Using sample file: {AUDIO_FILE}")

## 4. Load model and show device info

In [ ]:
import torch
from granite_asr.config import get_settings
from granite_asr.model import load_model

settings = get_settings()
device = settings.resolved_device
dtype = settings.resolved_dtype

print(f"Resolved device : {device}")
print(f"Resolved dtype  : {dtype}")
if device == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading model (this may take 1-3 minutes on first run)...")
import time
t0 = time.time()
load_model()
print(f"Model loaded in {time.time() - t0:.1f}s")

## 5. Run transcription

In [ ]:
from granite_asr import transcribe

with open(AUDIO_FILE, "rb") as f:
    audio_bytes = f.read()

print(f"Audio file size: {len(audio_bytes) / 1024:.1f} KB")
print("Running transcription...")

t0 = time.time()
result = transcribe(audio_bytes, language="en")  # change language as needed
elapsed = time.time() - t0

print(f"\nTranscription completed in {elapsed:.2f}s")
print(f"Audio duration  : {result.audio_duration_s:.1f}s")
print(f"Real-time factor: {result.audio_duration_s / elapsed:.2f}x")
print()
for seg in result.segments:
    print(f"[{seg.start:.2f}s – {seg.end:.2f}s] [{seg.speaker}] {seg.text}")

## 6. Benchmark: inference time over multiple runs

In [ ]:
import statistics

N_RUNS = 5
times = []

print(f"Running {N_RUNS} inference passes...")
for i in range(N_RUNS):
    t0 = time.time()
    transcribe(audio_bytes, language="en")
    elapsed = time.time() - t0
    times.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.2f}s")

print()
print(f"Mean  : {statistics.mean(times):.2f}s")
print(f"Median: {statistics.median(times):.2f}s")
print(f"Stdev : {statistics.stdev(times):.2f}s")
print(f"RTF   : {result.audio_duration_s / statistics.mean(times):.2f}x  (higher is faster)")